<!-- STANDIN notebook title -->
# Part notebook: supervised models and their explanations (steps A4–A6, B1, B2, B5, B6, C1, C2)

Owner: Kirill. This title and the stand-in cells are dropped by `tools/assemble.py`.

In [1]:
# STANDIN A1-A3
%run 00_setup.ipynb

(11430, 89)
status
legitimate    5715
phishing      5715
Name: count, dtype: int64


6858 2286 2286
train      phishing share: 0.500
validation phishing share: 0.500
test       phishing share: 0.500


<!-- STEP A4 -->
### 5.1 Supervised: all labels

## Step A4: Glass box 1: a small decision tree

We try depth 3 and 4 and keep the better one on the **validation** set (never on test). We read the
tree in step B1.

In [2]:
# STEP A4
from sklearn.tree import DecisionTreeClassifier, export_text

best_score, tree = -1, None
depth_scores = []                                  # both depths, for the report
for depth in [3, 4]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=42)
    t.fit(X_train, y_train)
    scores = report(t, X_val, y_val)
    score = scores["macro_F1"]                     # choose on VALIDATION
    print("depth", depth, "validation macro-F1:", round(score, 3))
    depth_scores.append({"max_depth": depth, "leaves": t.get_n_leaves(), **scores})
    if score > best_score:
        best_score, tree = score, t
print("kept depth", tree.get_depth())

depth_scores = pd.DataFrame(depth_scores)
depth_scores.to_csv("results/tables/A4_tree_depth.csv", index=False)
depth_scores

depth 3 validation macro-F1: 0.913


depth 4 validation macro-F1: 0.925
kept depth 4


,max_depth,leaves,macro_F1,recall,ROC_AUC,FAR
0,3,8,0.913,0.904,0.948,0.079
1,4,16,0.925,0.920,0.954,0.070


<!-- STEP A5 -->
## Step A5: Glass box 2: logistic regression

One weight per feature. We scale the features first so the weights can be compared; the `Pipeline`
fits the scaler on training data only.

In [3]:
# STEP A5
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# The Pipeline fits the scaler on training data only. No leakage.
logreg = Pipeline([
    ("scale", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000)),
])
logreg.fit(X_train, y_train)
print(report(logreg, X_val, y_val))

{'macro_F1': 0.938, 'recall': 0.933, 'ROC_AUC': 0.983, 'FAR': 0.056}


<!-- STEP A6 -->
## Step A6: The black box: a Random Forest

300 trees trained on all labels. This is the **upper line**: the best we can hope for when every URL
is labelled.

In [4]:
# STEP A6
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print(report(rf, X_val, y_val))

{'macro_F1': 0.961, 'recall': 0.968, 'ROC_AUC': 0.993, 'FAR': 0.046}
